# U-JEPA Phase 1: N-LoRA continual learning on Qwen3-14B-Instruct

Gate: average forgetting < 5 percent after sequential FOMC then ScienceQA-text training. Requires GPU T4 x2 and Internet On. Output: `/kaggle/working/results/phase1_continual.json`.

**Note on disk**: HF model cache goes to `/tmp/hf_cache` (50+ GB ephemeral) because `/kaggle/working` is capped at 20 GB and Qwen3-14B-Instruct is ~28 GB at fp16 download.

**Note on auth**: Qwen3-14B-Instruct may require a HuggingFace access token. Set the `HF_TOKEN` Kaggle secret before running. If it is missing the run will warn and continue; the model download will still work if the gate is currently off.

In [ ]:
# Cell 0: remove any old HF cache stuck inside /kaggle/working quota from a previous run
import shutil, os, subprocess
stale = '/kaggle/working/hf_cache'
if os.path.isdir(stale):
    print(f'removing stale cache at {stale}')
    shutil.rmtree(stale)
os.makedirs('/tmp/hf_cache', exist_ok=True)
print('df -h /kaggle/working /tmp:')
try:
    print(subprocess.check_output(['df', '-h', '/kaggle/working', '/tmp']).decode())
except Exception as e:
    print(f'df check skipped: {e}')


In [ ]:
import subprocess, os, sys
if not os.path.exists('/kaggle/working/U-JEPA'):
    subprocess.run(['git', 'clone', 'https://github.com/kartikshirode/U-JEPA.git',
                    '/kaggle/working/U-JEPA'], check=True)
else:
    subprocess.run(['git', '-C', '/kaggle/working/U-JEPA', 'pull'], check=True)
os.chdir('/kaggle/working/U-JEPA')
# Install Kaggle extras (loud, not -q, so pip failures are visible in the log).
# requirements-kaggle.txt pulls vllm and autoawq too; Phase 1 itself only needs
# bitsandbytes + transformers, but we share the file with Phase 0 for consistency.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                'requirements-kaggle.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)


In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle secret')
except Exception as e:
    print(f'Warning: no HF_TOKEN secret found ({e}).')
    print('Phase 1 uses Qwen3-14B-Instruct which may be gated on HuggingFace.')
    print('If model download fails with 401, add HF_TOKEN as a Kaggle secret and rerun.')
# Force HF cache outside /kaggle/working quota; Qwen3-14B is ~28 GB and /kaggle/working is 20 GB
os.environ['HF_HOME'] = '/tmp/hf_cache'
os.environ['HF_HUB_CACHE'] = '/tmp/hf_cache'
os.environ['TRANSFORMERS_CACHE'] = '/tmp/hf_cache'
print(f"HF_HOME={os.environ['HF_HOME']}")


In [ ]:
import torch
print(f'torch {torch.__version__}, cuda {torch.version.cuda}, GPUs: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}, '
          f'{torch.cuda.get_device_properties(i).total_memory // (1024**2)} MiB')
assert torch.cuda.device_count() >= 1, 'Phase 1 needs at least one CUDA GPU; check accelerator setting.'


In [ ]:
import subprocess, sys, os
# Propagate cache + token env vars to the subprocess
env = os.environ.copy()
env['HF_HOME'] = '/tmp/hf_cache'
env['HF_HUB_CACHE'] = '/tmp/hf_cache'
env['TRANSFORMERS_CACHE'] = '/tmp/hf_cache'
subprocess.run([sys.executable, 'scripts/02_train_continual_phase1.py'], check=True, env=env)


In [ ]:
import json
from pathlib import Path
p = Path('/kaggle/working/results/phase1_continual.json')
if p.exists():
    print(json.dumps(json.loads(p.read_text()), indent=2))
else:
    print('No results file yet')
log = Path('/kaggle/working/results/phase1_continual.log')
if log.exists():
    print('=== last 60 log lines ===')
    for line in log.read_text().splitlines()[-60:]:
        print(line)
